# 02 – Data Preprocessing

This notebook reproduces the preprocessing used by the final model. It removes exact duplicate rows, separates features and target, performs an 80/20 split, scales numerical columns, and one-hot encodes categorical columns. The preprocessor is fitted only on training data to avoid data leakage.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [ ]:
df = pd.read_csv("../Insurance.csv")
print("Original shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Shape after removing duplicates:", df.shape)
print("Missing values:\n", df.isnull().sum())

In [ ]:
X = df.drop(columns=["charges"])
y = df["charges"]
num_cols = ["age", "bmi", "children"]
cat_cols = ["sex", "smoker", "region"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
    ]
)
X_train_final = preprocessor.fit_transform(X_train)
X_test_final = preprocessor.transform(X_test)
print("Processed training shape:", X_train_final.shape)
print("Processed testing shape:", X_test_final.shape)
print("Processed feature names:", preprocessor.get_feature_names_out())

## Skewness check

Skewness is reviewed for context. The selected model is trained on the original `charges` target, matching the earlier model comparison. We do not apply a log transform in this final workflow.

In [ ]:
print("Feature skewness:")
print(X_train[num_cols].skew())
print("\nOriginal charges skewness:")
print(y_train.skew())